# 03 — Phase 2: Rare-Class Identification (Entropy Method)

Applies the reference paper's Definitions 1–2 (implemented in
`src/rare_class/entropy.py`) to the **trained baseline model's own
embeddings**, so "which class is rare" is measured, not assumed — and
so we can later check whether fusion (Phase 3) changes which class looks
rare, which would itself be an interesting result.

Assumes `02_baseline_signal_only.ipynb` has been run (needs
`experiments/results/baseline_signal_only_site1v2/best_model.pt`).

In [ ]:
%cd /content/mm-dg-healthcare
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.data.ptbxl_dataset import load_ptbxl_metadata, PTBXLDataset, DIAGNOSTIC_SUPERCLASSES
from src.data.domain_split import split_by_domain
from src.models.signal_encoder import SignalEncoder
from src.models.classifier import SignalOnlyClassifier
from src.rare_class.entropy import identify_rare_classes
from src.utils.config import load_config
from src.training.trainer import extract_embeddings
from torch.utils.data import DataLoader

cfg = load_config("configs/baseline_signal_only.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load the trained baseline and rebuild its train-domain loader

In [ ]:
metadata = load_ptbxl_metadata(cfg.data.ptbxl_dir)
split = split_by_domain(metadata, "site", cfg.data.train_sites, cfg.data.test_sites)
label_to_idx = {c: i for i, c in enumerate(DIAGNOSTIC_SUPERCLASSES)}
idx_to_label = {i: c for c, i in label_to_idx.items()}

train_ds = PTBXLDataset(split.train_df, cfg.data.ptbxl_dir, label_to_idx)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2)

encoder = SignalEncoder(embed_dim=cfg.model.embed_dim, channel_widths=tuple(cfg.model.channel_widths))
model = SignalOnlyClassifier(encoder, num_classes=len(label_to_idx), embed_dim=cfg.model.embed_dim, hidden_dim=cfg.model.get("hidden_dim")).to(device)
model.load_state_dict(torch.load("experiments/results/baseline_signal_only_site1v2/best_model.pt", map_location=device))
model.eval()
print("Loaded trained baseline.")

## Extract embeddings and compute per-class entropy

In [ ]:
embeddings, labels = extract_embeddings(model, train_loader, device)
print(f"Embeddings: {embeddings.shape}, labels: {labels.shape}")

result = identify_rare_classes(embeddings, labels, k=cfg.rare_class.k)

print("\nPer-class entropy (theta):")
for cls_idx, theta in sorted(result.class_entropy.items()):
    flag = " <-- RARE" if cls_idx in result.rare_classes else ""
    print(f"  {idx_to_label[cls_idx]:6s} theta={theta:.4f}{flag}")

print(f"\nMean entropy: {result.mean_entropy:.4f}, Std: {result.std_entropy:.4f}")
print(f"Rare classes (Definition 2 threshold = mean +/- 1 std): "
      f"{[idx_to_label[c] for c in result.rare_classes]}")

## Visualize

In [ ]:
labels_sorted = sorted(result.class_entropy.keys())
thetas = [result.class_entropy[c] for c in labels_sorted]
colors = ["crimson" if c in result.rare_classes else "steelblue" for c in labels_sorted]

plt.figure(figsize=(7, 4))
plt.bar([idx_to_label[c] for c in labels_sorted], thetas, color=colors)
plt.axhline(result.mean_entropy, color="gray", linestyle="--", label="mean")
plt.axhline(result.mean_entropy + result.std_entropy, color="gray", linestyle=":", label="mean ± 1 std")
plt.axhline(result.mean_entropy - result.std_entropy, color="gray", linestyle=":")
plt.ylabel("Class entropy (theta)")
plt.title("Per-class entropy — red bars are classes flagged as rare")
plt.legend()
plt.show()

## Compare against raw class frequency

Worth checking: is the "rare" class by entropy the same as the class with
fewest raw examples? If not, that's exactly the reference paper's point —
rarity isn't just about count, it's about how spread out/inconsistent the
class's representations are.

In [ ]:
counts = np.bincount(labels, minlength=len(label_to_idx))
for i, c in enumerate(counts):
    print(f"{idx_to_label[i]:6s} n={c:5d}  theta={result.class_entropy.get(i, float('nan')):.4f}")

## Next steps (not yet built)

This notebook establishes *which class is rare* and confirms the entropy
math (already unit-tested in `tests/test_entropy.py`) runs correctly on
real learned embeddings. Phases 3–6 (multimodal fusion, missing-modality
robustness, full ablation) are scaffolded in `src/models/fusion.py`,
`src/models/text_encoder.py`, and `src/training/modality_dropout.py` —
see each file's docstring for the implementation plan, and
`notebooks/04_multimodal_fusion.ipynb` onward for their placeholders.